# COVID-19 Detection using Tongue Images
Using MobileNetV3, EfficientNetV2, and ConvNeXt-T (PyTorch + timm)

In [ ]:
import pandas as pd
import os

# ✅ Step 1: Set path to your Excel file
excel_path = r"C:\Users\ajith\OneDrive\Desktop\Dissertation\dataset\content\Tonges.xlsx"

# ✅ Step 2: Read Excel and rename columns
df = pd.read_excel(excel_path)

# Print actual columns to check (for debugging)
print("Columns in Excel:", df.columns.tolist())

# Take first two columns only
df = df.iloc[:, :2]
df.columns = ['id', 'label']

# ✅ Step 3: Clean up ID and label
df['id'] = df['id'].astype(str).str.strip()
df['label'] = df['label'].astype(str).str.strip()

# ✅ Step 4: Build full image filenames like IMG_20220411_09249177.jpg
df['filename'] = 'IMG_20220411_' + df['id'] + '.jpg'

# ✅ Step 5: Keep only necessary columns
df = df[['filename', 'label']]

# Show result
df.head()


✅ Matched image files: 0 / 0


,filename,label


In [10]:
import os

image_dir = r"C:\Users\ajith\OneDrive\Desktop\Dissertation\dataset\content\dataset"

# List a few actual image files
actual_files = os.listdir(image_dir)
actual_files = [f for f in actual_files if f.endswith('.jpg')]
print("Sample image filenames:", actual_files[:5])


Sample image filenames: ['IMG_20220411_09235385.jpg', 'IMG_20220411_09235389.jpg', 'IMG_20220411_106246.jpg', 'IMG_20220411_106256.jpg', 'IMG_20220411_106258.jpg']


In [11]:
# Show a few generated filenames
print("Sample generated filenames:", df['filename'].head().tolist())


Sample generated filenames: []


In [ ]:
# ✅ Step 1: Install Required Libraries
!pip install torch torchvision timm scikit-learn matplotlib seaborn opencv-python pillow

## ✅ Step 2: Secondary Preprocessing

In [ ]:
import os
import cv2
import numpy as np
from PIL import Image
from pathlib import Path

input_base_dir = 'dataset/'
output_base_dir = 'processed/'
image_size = (224, 224)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

classes = ['COVID_Positive', 'COVID_Negative']
for cls in classes:
    Path(os.path.join(output_base_dir, cls)).mkdir(parents=True, exist_ok=True)

for cls in classes:
    input_dir = os.path.join(input_base_dir, cls)
    output_dir = os.path.join(output_base_dir, cls)
    for filename in os.listdir(input_dir):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(input_dir, filename)
            image = cv2.imread(img_path)
            if image is None:
                continue
            image = cv2.resize(image, image_size)
            lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
            l, a, b = cv2.split(lab)
            cl = clahe.apply(l)
            image = cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2BGR)
            image = image.astype(np.float32) / 255.0
            image_pil = Image.fromarray((image * 255).astype(np.uint8))
            image_pil.save(os.path.join(output_dir, filename))

## ✅ Step 3: Data Augmentation

In [ ]:
import torchvision.transforms as transforms

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

## ✅ Step 4: Load Dataset with Augmentation

In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

dataset = ImageFolder('processed', transform=train_transform)
train_len = int(0.7 * len(dataset))
val_len = int(0.15 * len(dataset))
test_len = len(dataset) - train_len - val_len
train_set, val_set, test_set = random_split(dataset, [train_len, val_len, test_len])

val_set.dataset.transform = val_test_transform
test_set.dataset.transform = val_test_transform

train_loader = DataLoader(train_set, batch_size=8, shuffle=True)
val_loader = DataLoader(val_set, batch_size=8)
test_loader = DataLoader(test_set, batch_size=1)

## ✅ Step 5: Train with `timm` Model (MobileNetV3 / EfficientNetV2 / ConvNeXt-T)

In [ ]:
import timm
import torch
import torch.nn as nn
import torch.optim as optim

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = timm.create_model('mobilenetv3_large_100', pretrained=True, num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(10):
    model.train()
    correct, total = 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f"Epoch {epoch+1}, Accuracy: {100 * correct / total:.2f}%")

## ✅ Step 6: Evaluate Model Performance

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns

model.eval()
y_true, y_pred, y_probs = [], [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(probs, 1)
        y_true.append(labels.item())
        y_pred.append(predicted.cpu().item())
        y_probs.append(probs[0][1].cpu().item())

print(classification_report(y_true, y_pred))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

fpr, tpr, _ = roc_curve(y_true, y_probs)
plt.plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_true, y_probs):.2f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()